# Model Training

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline


import pickle

In [3]:
from sklearn.preprocessing import LabelEncoder
import mlflow

In [4]:
from sklearn.model_selection import GridSearchCV

In [5]:
from mlflow.tracking import MlflowClient

In [6]:
df = pd.read_csv('shipment_price.csv')

In [7]:
df.sample(3)

,Line Item Quantity,Line Item Value,Pack Price,Country,Vendor,Product Group,Weight (Kilograms),Shipment Mode
4207,5,160.0,32.00,Botswana,"Trinity Biotech, Plc",HRDT,2,Air
7484,90000,729900.0,8.11,Mozambique,SCMS from RDC,ARV,3872,Truck
2022,2265,69807.3,30.82,Zambia,ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV),ARV,2272,Air


{
 'Air': 0,
 'Air Charter': 1,
 'Ocean': 2,
 'Truck': 3
}

In [8]:
df['Shipment Mode'].value_counts()/df['Shipment Mode'].count()*100

Air            66.650988
Truck          21.930856
Air Charter     7.584666
Ocean           3.833490
Name: Shipment Mode, dtype: float64

In [10]:
df.describe()

,Line Item Quantity,Line Item Value,Pack Price,Weight (Kilograms)
count,8504.000000,8.504000e+03,8504.000000,8504.000000
mean,20969.384290,1.792574e+05,20.690142,4102.173095
std,42242.282838,3.661193e+05,42.028265,12513.728428
min,1.000000,0.000000e+00,0.000000,0.000000
25%,800.000000,7.582345e+03,3.810000,387.000000
50%,4340.000000,4.408032e+04,8.260000,1638.500000
75%,21000.000000,2.000000e+05,20.480000,3653.250000
max,619999.000000,5.951990e+06,1250.000000,857354.000000


## Feature Engineering Pipeline Implement

In [51]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [52]:
X_train.shape, X_test.shape

((6803, 7), (1701, 7))

In [53]:
with open('fe_pipeline.pkl', 'rb') as file:
    loaded_pipeline = pickle.load(file)

In [54]:
loaded_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder_pipeline',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  ['Vendor', 'Product Group',
                                                   'Country']),
                                                 ('scaling_trf',
                                                  Pipeline(steps=[('tranform',
                                                                   PowerTransformer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Line Item Quantity',
                                                   'Line Item Value',
                                                   'Pack Price',
                                                   'Weight (Kilograms)'])]))])

In [66]:
X_train_trf = loaded_pipeline.fit_transform(X_train)
X_test_trf = loaded_pipeline.transform(X_test)

In [68]:
with open('fitted_pipeline.pkl','wb') as f:
    pickle.dump(loaded_pipeline, f)

## Target Encoding

In [122]:
df['Shipment Mode'].unique()

array(['Air', 'Truck', 'Air Charter', 'Ocean'], dtype=object)

In [69]:
le = LabelEncoder()

y_train_trf = le.fit_transform(y_train)
y_test_trf = le.transform(y_test)

## Model Evaluation

In [70]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [87]:
from sklearn.metrics import precision_score, f1_score, recall_score, confusion_matrix,accuracy_score,ConfusionMatrixDisplay,classification_report

In [72]:
import time

In [123]:
mlflow.set_tracking_uri('http://127.0.0.1:5000/')
mlflow.set_experiment('Shipment Mode Prediction')

models = {
    'LogisticRegression': LogisticRegression(),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'RandomForestClassifier' : RandomForestClassifier(),
    'GradientBoostingClassifier' : GradientBoostingClassifier(),
    'XGBClassifier': XGBClassifier()
}

results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):

        # model training and prediction
        start_time = time.time()
        model.fit(X_train_trf, y_train_trf)
        end_time = time.time()
        y_pred = model.predict(X_test_trf)

        # model metrics
        training_time = end_time - start_time
        precision = precision_score(y_test_trf, y_pred, average='weighted')
        recall = recall_score(y_test_trf, y_pred,average='weighted')
        f1 = f1_score(y_test_trf, y_pred,average='weighted')
        accuracy = accuracy_score(y_test_trf, y_pred)

        metrics = {'Training_Time':training_time,
                   'Precision':precision,
                   'Recall':recall,
                   'F1':f1,
                   'Accuracy':accuracy
                    }
        
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, artifact_path=name)
        mlflow.log_param('model_name',model)

        # confusion matrix
        cm = confusion_matrix(y_test_trf, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
        disp.plot(cmap='Blues',xticks_rotation=45)
        plt.title(f'Confusion Matrix - {name}')
        file_name = f'confusion_matrix_{name}.png'
        plt.savefig(file_name)
        mlflow.log_artifact(file_name)
        plt.close()

        # classification report
        report = classification_report(y_test_trf, y_pred)
        report_file = f'classification_report_{name}.txt'
        with open(report_file,'w') as f:
            f.write(report)
        mlflow.log_artifact(report_file)


        results[name] = metrics
        
results_df = pd.DataFrame(results).T
print('Model Performance Metrics:\n')
print(results_df)

2026/03/30 09:19:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 09:19:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/4/runs/62b2b7605acf48b89c3a2e8f985618fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/03/30 09:19:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 09:19:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTreeClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/d9757219b3334085886a837bdbac40dc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/03/30 09:19:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 09:19:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/aca16413003d420189571e3d9942a830
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/03/30 09:19:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 09:19:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run GradientBoostingClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/37c9c032a7c94871b1cf1abc2fe40a3c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/03/30 09:20:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 09:20:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/871a0963bf614d51942eea57efe1dc0b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Model Performance Metrics:

                            Training_Time  Precision    Recall        F1  \
LogisticRegression               0.371945   0.841017  0.835979  0.837611   
DecisionTreeClassifier           0.112736   0.826084  0.822457  0.823806   
RandomForestClassifier           2.134290   0.870714  0.870076  0.870291   
GradientBoostingClassifier      18.066233   0.863878  0.861846  0.862411   
XGBClassifier                    2.106925   0.878237  0.877131  0.877555   

                            Accuracy  
LogisticRegression          0.835979  
DecisionTreeClassifier      0.822457  
RandomForestClassifier      0.870076  
GradientBoostingClassifier  0.861846  
XGBClassifier               0.877131  


## Register Model in Mlflow

In [124]:
registered_model = 'final_model'
run_id = '871a0963bf614d51942eea57efe1dc0b'
final_model_name = 'XGBClassifier'

model_uri = f'runs:/{run_id}/{final_model_name}'

mlflow.register_model(model_uri=model_uri, name=registered_model)

Successfully registered model 'final_model'.
2026/03/30 09:27:35 WARNING mlflow.tracking._model_registry.fluent: Run with id 871a0963bf614d51942eea57efe1dc0b has no artifacts at artifact path 'XGBClassifier', registering model based on models:/m-103e82adb0fb4ee583f160d4dec86054 instead
2026/03/30 09:27:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: final_model, version 1
Created version '1' of model 'final_model'.


<ModelVersion: aliases=[], creation_timestamp=1774843056061, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774843056061, metrics=None, model_id=None, name='final_model', params=None, run_id='871a0963bf614d51942eea57efe1dc0b', run_link='', source='models:/m-103e82adb0fb4ee583f160d4dec86054', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

## Load Best Model From MLflow

In [125]:
registered_model = 'final_model'
model_version = 1

model_uri = f'models:/{registered_model}/{model_version}'

loaded_model_xgb = mlflow.sklearn.load_model(model_uri)

## Hyperparamter Tuning

In [128]:
params = {
    'n_estimators':[50,100,150,200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

grid_xgb = GridSearchCV(estimator=loaded_model_xgb,param_grid=params,cv=5,scoring='precision_weighted',n_jobs=-1)

grid_xgb.fit(X_train_trf, y_train_trf)

GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [50, 100, 150, 200]},
             scoring='precision_weighted')

In [129]:
grid_xgb.best_params_, grid_xgb.best_score_

({'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 100},
 0.8787625185948231)

In [ ]:
tuned_model = XGBClassifier(grid_xgb.best_params_)
tuned_model.fit(X_train_trf, y_train_trf)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

## Register Tuned Model As Version 2 in MLflow

In [140]:
grid_xgb.best_params_

{'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 100}

In [143]:
with mlflow.start_run(run_name='XGB_Tuned_Run'):
    # Log tuned model
    mlflow.sklearn.log_model(tuned_model, name='model')
    
    mlflow.log_params(grid_xgb.best_params_)

    # Log metrics
    y_pred = tuned_model.predict(X_test_trf)

    mlflow.log_metric("Precision", precision_score(y_test_trf, y_pred, average='weighted'))
    mlflow.log_metric("Recall", recall_score(y_test_trf, y_pred, average='weighted'))
    mlflow.log_metric("F1", f1_score(y_test_trf, y_pred, average='weighted'))
    mlflow.log_metric("Accuracy", accuracy_score(y_test_trf, y_pred))


2026/03/30 10:24:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_Tuned_Run at: http://127.0.0.1:5000/#/experiments/4/runs/8ec81a2f369b4cc7b3d20ba461251b49
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [137]:
client = MlflowClient()
run_id = '24a5ea8dd8f74f8c8f56674e13cad5bf'

# This will create the next version automatically
new_model_version = client.create_model_version(
    name=registered_model,                # existing registered model name
    source=f"runs:/{run_id}/model", # artifact path of this run
    run_id=run_id
)

2026/03/30 09:51:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: final_model, version 2


## Model Saving

In [139]:
with open('final_model.pkl','wb') as f:
    pickle.dump(tuned_model,f)